# Round15 全流程 Notebook：`v15_3_ta_refine_push.csv`

这个 notebook 是完整复现流程：从官方 GitHub 原始数据读取，到策略渲染、导出提交文件、诊断分析。

**当前分支配置**
- family: `nearboost_refine`
- ec_mode: `none`
- drp_mode: `control`
- alpha: `nan`
- near_alpha: `1.28`
- far_alpha: `0.88`
- power: `0.55`
- drp_weight: `0.0`
- drp_near_power: `nan`


## Step 1. Experiment Objective

**这个步骤做什么**
明确当前分支的策略假设、风险等级和验证目标。

**为什么要做这个步骤**
保证上传文件不是重复试错，而是有结构化信息增益。

**预期产出**
得到本分支的实验定位。

## Step 2. Environment Setup

**这个步骤做什么**
导入依赖并初始化路径与数据源。

**为什么要做这个步骤**
保证 notebook 可复现。

**预期产出**
得到统一运行环境。

In [ ]:
# =========================
# Step: Environment Setup
# 作用：导入依赖，定义路径和官方数据源
# =========================

# 标准库
import json
import subprocess
from io import StringIO
from pathlib import Path

# 数据与数值
import numpy as np
import pandas as pd

# 可视化
import matplotlib.pyplot as plt

# 机器学习组件
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.neighbors import NearestNeighbors

# 显示设置
pd.set_option('display.max_columns', 260)
pd.set_option('display.width', 220)

# 本地路径
PROJECT_ROOT = Path('/Users/zhangziling/Documents/New project')
ROUND_DIR = PROJECT_ROOT / 'round15'

# 官方 GitHub 原始数据地址
BASE_RAW = 'https://raw.githubusercontent.com/Snowflake-Labs/EY-AI-and-Data-Challenge/refs/heads/main'

## Step 3. Load Official Raw Data

**这个步骤做什么**
从官方 GitHub 读取原始训练/验证数据。

**为什么要做这个步骤**
保证数据来源一致且可追溯。

**预期产出**
得到 6 张官方原始表。

In [ ]:
# =========================
# Step: Load Official Raw Data From GitHub
# 作用：读取挑战赛官方原始训练/验证数据
# =========================

def read_csv_github(url: str) -> pd.DataFrame:
    # 优先 pandas 直读 URL；失败时自动回退到 curl。
    try:
        return pd.read_csv(url)
    except Exception as e:
        print(f'[warn] pd.read_csv failed, fallback to curl: {url}')
        print(f'       reason: {e}')
        res = subprocess.run(['curl', '-L', url], check=True, capture_output=True, text=True)
        return pd.read_csv(StringIO(res.stdout))

# 训练三表
train_wq = read_csv_github(f'{BASE_RAW}/water_quality_training_dataset.csv')
train_ls = read_csv_github(f'{BASE_RAW}/landsat_features_training.csv')
train_tc = read_csv_github(f'{BASE_RAW}/terraclimate_features_training.csv')

# 验证三表
val_tpl = read_csv_github(f'{BASE_RAW}/submission_template.csv')
val_ls = read_csv_github(f'{BASE_RAW}/landsat_features_validation.csv')
val_tc = read_csv_github(f'{BASE_RAW}/terraclimate_features_validation.csv')

print('train_wq:', train_wq.shape)
print('train_ls:', train_ls.shape)
print('train_tc:', train_tc.shape)
print('val_tpl:', val_tpl.shape)
print('val_ls:', val_ls.shape)
print('val_tc:', val_tc.shape)

## Step 4. Data QA

**这个步骤做什么**
检查结构与缺失率，确认关键字段可用。

**为什么要做这个步骤**
提前发现问题，减少后续调试成本。

**预期产出**
得到基础质量报告。

In [ ]:
# =========================
# Step: Data QA
# 作用：检查形状、列名、缺失率
# =========================

def qa_table(df: pd.DataFrame, name: str, topn: int = 10):
    print(f'\n===== {name} =====')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    miss = df.isna().mean().sort_values(ascending=False).head(topn)
    print('top missing ratio:')
    print(miss)

qa_table(train_wq, 'train_wq')
qa_table(train_ls, 'train_ls')
qa_table(train_tc, 'train_tc')
qa_table(val_tpl, 'val_tpl')

display(train_wq.head(3))
display(val_tpl.head(3))

## Step 5. Merge And Feature Prep

**这个步骤做什么**
融合多源数据并构建基础特征。

**为什么要做这个步骤**
形成完整分析上下文，保证全流程完整。

**预期产出**
得到 train_full/val_full 及基础特征版本。

In [ ]:
# =========================
# Step: Merge Data + Basic Features
# 作用：合并多源表并生成基础时间/空间特征
# =========================

KEY = ['Latitude', 'Longitude', 'Sample Date']
for df in [train_wq, train_ls, train_tc, val_tpl, val_ls, val_tc]:
    df['Sample Date'] = pd.to_datetime(df['Sample Date'], dayfirst=True, errors='coerce')

train_full = (
    train_wq
    .merge(train_ls, on=KEY, how='left', suffixes=('', '_ls'))
    .merge(train_tc, on=KEY, how='left', suffixes=('', '_tc'))
    .copy()
)

val_full = (
    val_tpl
    .merge(val_ls, on=KEY, how='left', suffixes=('', '_ls'))
    .merge(val_tc, on=KEY, how='left', suffixes=('', '_tc'))
    .copy()
)

print('train_full:', train_full.shape)
print('val_full:', val_full.shape)

def add_basic_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    dt = out['Sample Date']

    # 时间循环特征
    out['month'] = dt.dt.month.astype(float)
    out['dayofyear'] = dt.dt.dayofyear.astype(float)
    out['month_sin'] = np.sin(2 * np.pi * out['month'] / 12.0)
    out['month_cos'] = np.cos(2 * np.pi * out['month'] / 12.0)
    out['doy_sin'] = np.sin(2 * np.pi * out['dayofyear'] / 366.0)
    out['doy_cos'] = np.cos(2 * np.pi * out['dayofyear'] / 366.0)

    # 空间交互
    out['lat_abs'] = out['Latitude'].abs()
    out['lon_abs'] = out['Longitude'].abs()
    out['lat_x_lon'] = out['Latitude'] * out['Longitude']
    return out

train_feat = add_basic_features(train_full)
val_feat = add_basic_features(val_full)
print('train_feat:', train_feat.shape)
print('val_feat:', val_feat.shape)

## Step 6. Define Strategy Helpers

**这个步骤做什么**
定义 TA/EC/DRP 策略函数与 DRP feature-seeded 生成函数。

**为什么要做这个步骤**
round15 的核心创新在 DRP feature-seeded 路线，需清晰可复现。

**预期产出**
得到可复用策略函数库。

In [ ]:
# =========================
# Step: Strategy Helper Functions
# 作用：定义 round15 的 TA/EC/DRP策略函数
# =========================

def compute_scale(train_coords: pd.DataFrame, val_coords: pd.DataFrame) -> np.ndarray:
    tr_unique = train_coords[['Latitude', 'Longitude']].drop_duplicates().to_numpy()
    va = val_coords[['Latitude', 'Longitude']].to_numpy()
    nn = NearestNeighbors(n_neighbors=1, metric='euclidean').fit(tr_unique)
    dist, _ = nn.kneighbors(va)
    dist = dist.reshape(-1)
    q10, q90 = np.quantile(dist, [0.10, 0.90])
    return np.clip((dist - q10) / max(q90 - q10, 1e-9), 0.0, 1.0)


def ta_global(v2_ta: np.ndarray, delta_ctrl_ta: np.ndarray, alpha: float) -> np.ndarray:
    return np.clip(v2_ta + alpha * delta_ctrl_ta, 0, None)


def ta_nearboost(v2_ta, delta_ctrl_ta, scale, near_alpha, far_alpha, power):
    alpha = far_alpha + (near_alpha - far_alpha) * np.power(1.0 - scale, power)
    return np.clip(v2_ta + alpha * delta_ctrl_ta, 0, None)


def ec_adjust(ec_control, ec_v2, scale, mode):
    if mode == 'none':
        return ec_control.copy()
    if mode == 'mild':
        w = 0.015 + 0.025 * np.clip((scale - 0.72) / 0.28, 0, 1)
        out = (1.0 - w) * ec_control + w * ec_v2
        return np.clip(out, 0, None)
    raise ValueError(mode)


def build_drp_feature_prediction(train_full_df, val_full_df):
    # 这个函数是 round15 新增重点：通过原始特征训练 DRP 模型，生成 feature-seeded 信号。
    tr = train_full_df.copy()
    va = val_full_df.copy()

    def add_features(df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        dt = out['Sample Date']
        out['year'] = dt.dt.year.astype(float)
        out['month'] = dt.dt.month.astype(float)
        out['dayofyear'] = dt.dt.dayofyear.astype(float)
        out['month_sin'] = np.sin(2 * np.pi * out['month'] / 12.0)
        out['month_cos'] = np.cos(2 * np.pi * out['month'] / 12.0)
        out['doy_sin'] = np.sin(2 * np.pi * out['dayofyear'] / 366.0)
        out['doy_cos'] = np.cos(2 * np.pi * out['dayofyear'] / 366.0)
        out['lat_abs'] = out['Latitude'].abs()
        out['lon_abs'] = out['Longitude'].abs()
        out['lat_x_lon'] = out['Latitude'] * out['Longitude']

        eps = 1e-6
        if {'nir', 'green'}.issubset(out.columns):
            out['ndvi'] = (out['nir'] - out['green']) / (out['nir'] + out['green'] + eps)
        if {'swir16', 'nir'}.issubset(out.columns):
            out['ndbi'] = (out['swir16'] - out['nir']) / (out['swir16'] + out['nir'] + eps)
        if {'swir22', 'swir16'}.issubset(out.columns):
            out['swir_ratio'] = out['swir22'] / (out['swir16'] + eps)
        if {'pet', 'NDMI'}.issubset(out.columns):
            out['pet_x_ndmi'] = out['pet'] * out['NDMI']
        if {'pet', 'MNDWI'}.issubset(out.columns):
            out['pet_x_mndwi'] = out['pet'] * out['MNDWI']
        return out

    tr = add_features(tr)
    va = add_features(va)

    key = ['Latitude', 'Longitude', 'Sample Date']
    ignore = set(key + ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])
    feat_cols = [c for c in tr.columns if c in va.columns and c not in ignore]

    Xtr = tr[feat_cols]
    Xva = va[feat_cols]
    y = np.log1p(np.clip(tr['Dissolved Reactive Phosphorus'].to_numpy(dtype=float), 0, None))

    imp = SimpleImputer(strategy='median')
    Xtr_i = imp.fit_transform(Xtr)
    Xva_i = imp.transform(Xva)

    model = ExtraTreesRegressor(
        n_estimators=900,
        max_depth=None,
        min_samples_leaf=2,
        max_features='sqrt',
        random_state=42,
        n_jobs=-1,
    )
    model.fit(Xtr_i, y)
    pred = np.expm1(model.predict(Xva_i))
    return np.clip(pred, 0, None)


def drp_adjust(drp_control, drp_probe, drp_feature, scale, mode, weight, near_power):
    if mode == 'control':
        return drp_control.copy()

    if mode in {'micro_uniform', 'micro_near'}:
        base = drp_probe
    elif mode in {'feature_seed_uniform', 'feature_seed_near'}:
        base = drp_feature
    else:
        raise ValueError(mode)

    if mode.endswith('_uniform'):
        w = np.full_like(scale, fill_value=weight, dtype=float)
    else:
        w = weight * np.power(1.0 - scale, near_power)

    return np.clip(drp_control + w * (base - drp_control), 0, None)

## Step 7. Load Anchors

**这个步骤做什么**
加载 v2、control、DRP probe，并构建 DRP feature seed。

**为什么要做这个步骤**
基于稳定锚点做小半径迭代，同时引入 DRP 新信号。

**预期产出**
得到渲染所需核心数组。

In [ ]:
# =========================
# Step: Load Anchors And Build DRP Feature Seed
# 作用：准备 round15 策略渲染的核心数组
# =========================

v2_ref = pd.read_csv(PROJECT_ROOT / 'submission_template_v2.csv')
control_ref = pd.read_csv(PROJECT_ROOT / 'round14/v14_3_ta_refine_push.csv')
drp_probe_ref = pd.read_csv(PROJECT_ROOT / 'v10_5_ec_ta_drp_link.csv')

assert len(v2_ref) == len(val_tpl), 'v2_ref length mismatch'
assert len(control_ref) == len(val_tpl), 'control_ref length mismatch'
assert len(drp_probe_ref) == len(val_tpl), 'drp_probe_ref length mismatch'

# DRP feature-seeded 预测（round15新增）
drp_feature_pred = build_drp_feature_prediction(train_full, val_full)

scale_val = compute_scale(train_wq[['Latitude', 'Longitude']], val_tpl[['Latitude', 'Longitude']])

v2_ta = v2_ref['Total Alkalinity'].to_numpy(dtype=float)
v2_ec = v2_ref['Electrical Conductance'].to_numpy(dtype=float)
v2_drp = v2_ref['Dissolved Reactive Phosphorus'].to_numpy(dtype=float)

ctrl_ta = control_ref['Total Alkalinity'].to_numpy(dtype=float)
ctrl_ec = control_ref['Electrical Conductance'].to_numpy(dtype=float)
ctrl_drp = control_ref['Dissolved Reactive Phosphorus'].to_numpy(dtype=float)

probe_drp = drp_probe_ref['Dissolved Reactive Phosphorus'].to_numpy(dtype=float)

delta_ctrl_ta = ctrl_ta - v2_ta

print('anchors ready.')
print('scale min/max:', float(scale_val.min()), float(scale_val.max()))
print('mean |TA control-v2|:', float(np.mean(np.abs(delta_ctrl_ta))))
print('mean |DRP probe-control|:', float(np.mean(np.abs(probe_drp - ctrl_drp))))
print('mean |DRP feature-control|:', float(np.mean(np.abs(drp_feature_pred - ctrl_drp))))

## Step 8. Render Strategy And Export

**这个步骤做什么**
按当前参数渲染预测并导出 csv。

**为什么要做这个步骤**
确保每个分支可独立复现和上传。

**预期产出**
得到 `v15_3_ta_refine_push.csv`。

In [ ]:
# =========================
# Step: Render Strategy And Export CSV
# 作用：按当前分支参数生成最终提交文件
# =========================

cfg = json.loads(r'''{
  "family": "nearboost_refine",
  "alpha": NaN,
  "near_alpha": 1.28,
  "far_alpha": 0.88,
  "power": 0.55,
  "ec_mode": "none",
  "drp_mode": "control",
  "drp_weight": 0.0,
  "drp_near_power": NaN,
  "ta_shift_vs_v2": 11.08605715782575,
  "ec_shift_vs_v2": 103.15716355682005,
  "drp_shift_vs_v2": 0.0,
  "ta_shift_vs_control": 2.1488627945067584,
  "ec_shift_vs_control": 0.0,
  "drp_shift_vs_control": 0.0,
  "ta_near_effect": 9.490027121106166,
  "ta_far_effect": 1.5960300367195839,
  "ta_out_ratio_1_99": 0.0,
  "ta_out_ratio_0p5_99p5": 0.0,
  "proxy_score": 7.94187798548817,
  "file_name": "v15_3_ta_refine_push.csv",
  "label": "ta_refine_push"
}''')

def clean_num(x):
    if x is None:
        return None
    try:
        return None if pd.isna(x) else float(x)
    except Exception:
        return x

family = cfg.get('family')
ec_mode = cfg.get('ec_mode', 'none')
drp_mode = cfg.get('drp_mode', 'control')

alpha = clean_num(cfg.get('alpha'))
near_alpha = clean_num(cfg.get('near_alpha'))
far_alpha = clean_num(cfg.get('far_alpha'))
power = clean_num(cfg.get('power'))
drp_weight = clean_num(cfg.get('drp_weight'))
drp_near_power = clean_num(cfg.get('drp_near_power'))

print('family:', family)
print('ec_mode:', ec_mode, '| drp_mode:', drp_mode)
print('alpha:', alpha, '| near_alpha:', near_alpha, '| far_alpha:', far_alpha, '| power:', power)
print('drp_weight:', drp_weight, '| drp_near_power:', drp_near_power)

# 1) TA
if family == 'control':
    ta = ctrl_ta.copy()
elif family == 'global_refine':
    if alpha is None:
        raise ValueError('global_refine needs alpha')
    ta = ta_global(v2_ta, delta_ctrl_ta, alpha=alpha)
elif family == 'nearboost_refine':
    if (near_alpha is None) or (far_alpha is None) or (power is None):
        raise ValueError('nearboost_refine needs near_alpha/far_alpha/power')
    ta = ta_nearboost(v2_ta, delta_ctrl_ta, scale_val, near_alpha, far_alpha, power)
else:
    raise ValueError(f'Unknown family: {family}')

# 2) EC
ec = ec_adjust(ctrl_ec, v2_ec, scale_val, mode=ec_mode)

# 3) DRP
drp = drp_adjust(
    drp_control=ctrl_drp,
    drp_probe=probe_drp,
    drp_feature=drp_feature_pred,
    scale=scale_val,
    mode=drp_mode,
    weight=0.0 if drp_weight is None else float(drp_weight),
    near_power=1.0 if drp_near_power is None else float(drp_near_power),
)

# 4) 导出
sub = val_tpl[['Latitude', 'Longitude', 'Sample Date']].copy()
sub['Sample Date'] = pd.to_datetime(sub['Sample Date']).dt.strftime('%d/%m/%Y')
sub['Total Alkalinity'] = np.clip(ta, 0, None)
sub['Electrical Conductance'] = np.clip(ec, 0, None)
sub['Dissolved Reactive Phosphorus'] = np.clip(drp, 0, None)
sub = sub[['Latitude', 'Longitude', 'Sample Date', 'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']]

out_file = ROUND_DIR / 'v15_3_ta_refine_push.csv'
sub.to_csv(out_file, index=False)
print('saved:', out_file)
print('shape:', sub.shape)

# 5) 诊断
diag = {
    'ta_shift_vs_v2': float(np.mean(np.abs(sub['Total Alkalinity'] - v2_ref['Total Alkalinity']))),
    'ec_shift_vs_v2': float(np.mean(np.abs(sub['Electrical Conductance'] - v2_ref['Electrical Conductance']))),
    'drp_shift_vs_v2': float(np.mean(np.abs(sub['Dissolved Reactive Phosphorus'] - v2_ref['Dissolved Reactive Phosphorus']))),
    'ta_shift_vs_control': float(np.mean(np.abs(sub['Total Alkalinity'] - control_ref['Total Alkalinity']))),
    'ec_shift_vs_control': float(np.mean(np.abs(sub['Electrical Conductance'] - control_ref['Electrical Conductance']))),
    'drp_shift_vs_control': float(np.mean(np.abs(sub['Dissolved Reactive Phosphorus'] - control_ref['Dissolved Reactive Phosphorus']))),
}

print('diagnostics:')
for k, v in diag.items():
    print(f'  - {k}: {v:.6f}')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(sub['Total Alkalinity'], bins=45, color='steelblue', alpha=0.85)
axes[0].set_title('TA Distribution')
axes[1].hist(sub['Electrical Conductance'], bins=45, color='darkorange', alpha=0.85)
axes[1].set_title('EC Distribution')
axes[2].hist(sub['Dissolved Reactive Phosphorus'], bins=45, color='seagreen', alpha=0.85)
axes[2].set_title('DRP Distribution')
plt.tight_layout()
plt.show()

sub.head()

## Step 9. Result Interpretation

**这个步骤做什么**
解释本分支策略目标、风险位置以及与 control 的差异。

**为什么要做这个步骤**
你每天上传次数有限，必须确保每个文件在实验设计上可解释且互补。

**本分支摘要**
- 输出文件：`v15_3_ta_refine_push.csv`
- family：`nearboost_refine`
- ec_mode：`none`
- drp_mode：`control`
- 重点看：`ta_shift_vs_control`、`drp_shift_vs_control`、分布稳定性。